In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Marketing campaigns (wide → long)

Quick reminder — `pd.wide_to_long`:
```python
pd.wide_to_long(
    df,
    stubnames=['clicks', 'revenue'],  # shared prefixes
    i='campaign_id',                  # row identifier
    j='quarter',                      # new column for the suffix
    suffix=r'\w+',                    # matches Q1, Q2, Q3...
    sep='_',                          # separator between stub and suffix
).reset_index()
```

- Reshape `campaigns` from wide to long. Each row should represent one campaign × quarter combination.
- Which quarter had the highest total revenue across all campaigns?
- Compute a `rev_per_click` column. Which campaign had the highest average across all quarters?
- Use `np.argmax` to find the index of the single highest-clicks row.

In [10]:
campaigns = pd.DataFrame({
    'campaign_id': ['C01','C02','C03'],
    'channel':     ['Email','Social','Search'],
    'clicks_Q1':   [1200, 3400, 2100],
    'clicks_Q2':   [1500, 2900, 2800],
    'clicks_Q3':   [1100, 3800, 3200],
    'revenue_Q1':  [4800, 8500, 6300],
    'revenue_Q2':  [6000, 7200, 8400],
    'revenue_Q3':  [4400, 9500, 9600],
})

# Your code here

w = pd.wide_to_long(
    campaigns,
    stubnames=['clicks', 'revenue'],
    i = 'campaign_id',
    j = 'quarter',
    suffix = r'\w+',
    sep ='_'
).reset_index()

print(w.groupby('quarter')['revenue'].sum().idxmax(),'had the highest total rev')

w['rev_per_click'] = w['revenue']/w['clicks']
print(w.groupby('campaign_id')['rev_per_click'].mean().idxmax(),'had the highest average')

print(w.iloc[np.argmax(w['clicks'])])


Q3 had the highest total rev
C01 had the highest average
campaign_id         C02
quarter              Q3
channel          Social
clicks             3800
revenue            9500
rev_per_click       2.5
Name: 7, dtype: object


---

## Level 2 — GDP and employment (merge_ordered)

Quick reminder — `pd.merge_ordered`:
```python
pd.merge_ordered(left, right, on='date', fill_method='ffill')
# Merges on a sorted key; fill_method='ffill' forward-fills NaN values
# introduced by the outer join
```

`gdp` is quarterly (Jan, Apr, Jul, Oct). `employment` is monthly. Merging them means GDP values need to be carried forward into the in-between months.

1. Use `pd.merge_ordered` with `fill_method='ffill'` to align the two tables into a monthly DataFrame.
2. How many rows have a forward-filled GDP value (i.e. were not originally in `gdp`)?
3. Is there a correlation between GDP growth and unemployment rate? Use `np.corrcoef`.
4. Use `ewm(span=3).mean()` to smooth the unemployment rate. Which month has the lowest smoothed value?

In [17]:
gdp = pd.DataFrame({
    'date':       pd.to_datetime(['2024-01-01','2024-04-01','2024-07-01','2024-10-01']),
    'gdp_growth': [2.1, 2.4, 2.8, 2.3],
})

employment = pd.DataFrame({
    'date': pd.to_datetime(['2024-01-01','2024-02-01','2024-03-01','2024-04-01',
                            '2024-05-01','2024-06-01','2024-07-01','2024-08-01',
                            '2024-09-01','2024-10-01','2024-11-01','2024-12-01']),
    'unemployment_rate': [3.8, 3.7, 3.6, 3.5, 3.4, 3.3, 3.4, 3.3, 3.2, 3.3, 3.2, 3.1],
})

# Your code here


m = pd.merge_ordered(
    gdp, 
    employment,
    on = 'date',
    fill_method= 'ffill'
)

print(m.shape[0] - gdp.shape[0],'were not originally in gdp')

cors = np.corrcoef(m['gdp_growth'],m['unemployment_rate'])[0,1]
print('correlation is ', cors)
print('moderately negatively associated')

m['sr'] = m['unemployment_rate'].ewm(span = 3).mean()

print(m.loc[m['sr'].idxmin(),'date'],'had the lowest smoothed value')

8 were not originally in gdp
correlation is  -0.5284229075567874
moderately negatively associated
2024-12-01 00:00:00 had the lowest smoothed value


---

## Level 3 — Retail products and transactions

Two tables: a products catalog and a sales log. No steps — clean, merge, and answer the five questions.

Known issues in `products`: `category` has inconsistent casing, `cost` is stored as `'$xx.xx'` strings.
Known issues in `transactions`: `channel` has inconsistent casing, one exact duplicate row.

1. Clean both tables and merge on `product_id`.
2. Add a `margin` column: `(price - cost) / price`. Use `.apply()` to add a `margin_tier`: margin ≥ 0.5 → `'High'`, ≥ 0.3 → `'Medium'`, else `'Low'`.
3. Use named aggregation to summarize by `category`: total revenue (`qty × price`), average margin, number of transactions.
4. Which channel drives the highest average revenue per transaction?
5. What are the 25th and 75th percentile margins? Use `np.percentile`.

In [35]:
products = pd.DataFrame({
    'product_id': ['P01','P02','P03','P04','P05','P06'],
    'name':       ['Wireless Mouse','USB Hub','Desk Lamp','Keyboard','Monitor','Webcam'],
    'category':   ['Electronics','Electronics','HOME OFFICE','electronics','Home Office','ELECTRONICS'],
    'cost':       ['$12.50','$8.00','$18.00','$22.00','$95.00','$15.00'],
    'price':      [29.99, 19.99, 34.99, 49.99, 199.99, 39.99],
})

transactions = pd.DataFrame({
    'txn_id':     ['T01','T02','T03','T04','T05','T06','T07','T08','T09','T10','T11','T02'],
    'product_id': ['P01','P02','P03','P01','P04','P05','P02','P06','P03','P05','P01','P02'],
    'qty':        [3, 5, 2, 1, 4, 1, 8, 2, 3, 2, 5, 5],
    'channel':    ['Online','IN-STORE','Online','in-store','Online','Online',
                   'IN-STORE','online','IN-STORE','Online','in-store','IN-STORE'],
})

# Your code here


products['category'] = products['category'].str.lower()
products['cost'] = products['cost'].str.replace('$','', regex = False)
products['cost'] = pd.to_numeric(products['cost'], errors='coerce')

transactions['channel'] = transactions['channel'].str.lower()
transactions = transactions.drop_duplicates().copy()


m = pd.merge(
    transactions, 
    products, 
    on = 'product_id',
    how = 'inner'
)

m['margin'] = (m['price'] - m['cost'])/m['price']
m['margin_tier'] = m['margin'].apply(lambda x: 'High' if x>=0.5 else
                                     'Medium' if x>=0.3 else
                                     'Low')
m['rev'] = m['price']* m['qty']

g = m.groupby('category').agg(
    total_rev = ('rev','sum'),
    avg_margin = ('margin','mean'),
    num_transc = ('txn_id','count')
)
print(g)

print(m.groupby('channel')['rev'].mean().idxmax(),'drives the highest average revenue per transaction')
print(np.percentile(m['margin'],[25,75]))

             total_rev  avg_margin  num_transc
category                                      
electronics     809.72    0.590572           7
home office     774.92    0.505272           4
online drives the highest average revenue per transaction
[0.52497625 0.59149715]
